# 05 · M4/M5 exploratory AI augmentation

This notebook loads the completed `ai_unreviewed_001` run. It uses 180 AI-generated paraphrases without human review, following the user's explicit instruction to generate and directly use them. The original human-reviewed protocol and course permission remain unresolved. No record is labelled human-approved. This notebook does not refit models or evaluate the final Test set.

In [1]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
if not (ROOT / "config.json").exists():
    ROOT = ROOT.parent
assert (ROOT / "config.json").exists()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
import pandas as pd
import joblib
RUN_DIR = ROOT / "results/augmentation_dev/ai_unreviewed_001"
run = json.loads((RUN_DIR / "run.json").read_text())

## Data provenance and shared training input

M4 and M5 fit the exact same augmented Train table. Each inherits its original baseline's feature design and searches only C on the fixed Dev split. All original artifacts remain at their existing paths.

In [2]:
{key: run[key] for key in ["augmentation_policy", "original_train_n", "paraphrase_n", "augmented_train_n", "dev_n", "same_augmented_table_for_M4_M5", "human_reviewed_protocol_satisfied", "test_evaluated"]}

{'augmentation_policy': 'ai_exploratory',
 'original_train_n': 14158,
 'paraphrase_n': 180,
 'augmented_train_n': 14338,
 'dev_n': 1880,
 'same_augmented_table_for_M4_M5': True,
 'human_reviewed_protocol_satisfied': False,
 'test_evaluated': False}

In [3]:
examples = pd.read_csv("data/paraphrases/train_paraphrases_ai_generated.csv", keep_default_na=False)
examples[["original_text", "paraphrase_text", "label", "origin", "review_status"]].head(6)

,original_text,paraphrase_text,label,origin,review_status
0,What is (are) Infertility ?,Could you explain what Infertility means?,Basic information,ai_generated,pending
1,what is the history of hps for Hantavirus ?,What is known about the history of HPS in rela...,Basic information,ai_generated,pending
2,what is botulism?,Could you explain what Botulism is?,Basic information,ai_generated,pending
3,what is yersiniosis for Yersinia ?,Could you explain yersiniosis as it relates to...,Basic information,ai_generated,pending
4,what is the government doing about these disea...,What action is the government taking on diseas...,Basic information,ai_generated,pending
5,what is holmes-adie syndrome ?,What is meant by Holmes-Adie syndrome?,Basic information,ai_generated,pending


## Actual development results

M4 and M5 tie the existing baselines on the original Dev set. These results do not establish improvement on alternative wording or real patient questions.

In [4]:
pd.read_csv(RUN_DIR / "comparison_M2_M3_M4_M5.csv")

,model_id,C,train_n,n,accuracy,macro_f1,augmentation_policy
0,M2,1.0,14158,1880,0.999468,0.999208,original_train_only
1,M3,0.1,14158,1880,0.999468,0.999208,original_train_only
2,M4,0.1,14338,1880,0.999468,0.999208,ai_exploratory
3,M5,0.1,14338,1880,0.999468,0.999208,ai_exploratory


In [5]:
pd.read_csv(RUN_DIR / "tuning.csv")

,model_id,C,n,accuracy,macro_f1
0,M4,0.1,1880,0.999468,0.999208
1,M4,1.0,1880,0.999468,0.999208
2,M4,10.0,1880,0.999468,0.999208
3,M5,0.1,1880,0.999468,0.999208
4,M5,1.0,1880,0.999468,0.999208
5,M5,10.0,1880,0.999468,0.999208


## Inference with saved pipelines

The following self-authored example is a demonstration input, not an additional experimental test set. Prediction returns only an intent category.

In [6]:
question = "What treatment options are available for asthma?"
outputs = []
for model_id in ["M4", "M5"]:
    estimator = joblib.load(ROOT / run["models"][model_id]["path"])
    outputs.append({"model_id": model_id, "question": question, "predicted_intent": estimator.predict([question])[0]})
pd.DataFrame(outputs)

,model_id,question,predicted_intent
0,M4,What treatment options are available for asthma?,Treatment
1,M5,What treatment options are available for asthma?,Treatment


## Development errors

The same topic-title ambiguity remains in both augmented models. Human error categorization is pending; the original label is preserved.

In [7]:
pd.read_csv(RUN_DIR / "error_analysis_pending.csv")

,question_id,question_text,label,disease_group,split,model_id,prediction,augmentation_policy,human_error_category,human_notes
0,10_MPlus_ADAM_QA/0002030.xml::0002030-1,What is (are) How to prevent frostbite and hyp...,Basic information,how to prevent frostbite and hypothermia,dev,M4,Prevention,ai_exploratory,NaN,NaN
1,10_MPlus_ADAM_QA/0002030.xml::0002030-1,What is (are) How to prevent frostbite and hyp...,Basic information,how to prevent frostbite and hypothermia,dev,M5,Prevention,ai_exploratory,NaN,NaN


See `docs/AI_AUGMENTATION_RUN.md` for paths, reproducible commands, limitations, and the distinction between exploratory AI augmentation and the originally planned human-reviewed experiment.